# GLUE Benchmark Evaluation & Results Analysis

This notebook aggregates results from multiple fine-tuning runs across GLUE tasks and compares DistilBERT and BERT with the values from the paper. 

This evaluation notebook provides:

1. **Results Aggregation**: Loads all fine-tuning results from multiple seeds
2. **Summary Statistics**: Median, mean, std, min, max for each task
3. **Model Comparison**: Side-by-side comparison of different models
4. **Paper Comparison**: Validates results against published DistilBERT paper results
5. **Result Export**: Consolidated JSON for archiving and sharing


## Setup: Import Libraries and Load Results

Should already be set up correct. 

In [2]:
import json
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

# Add project to path
project_root = Path().resolve().parent ##.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from distilbert_repro.glue_benchmark import GLUE_TASK_CONFIG, TaskConfig

print("Libraries imported successfully")

Libraries imported successfully


## Step 1: Locate and Load Results Files

Find all benchmark results from the finetuning runs.

In [3]:
# Find results files
results_dir = Path("./outputs/glue_benchmark")
results_files = sorted(results_dir.glob("results_*.json"))

print(f"Found {len(results_files)} results files:")
for rf in results_files:
    print(f"  - {rf.name}")

if not results_files:
    print("\nNo results files found!")
    print("Please run notebook 03-finetune-glue.ipynb first to generate results.")


Found 2 results files:
  - results_bert-base-uncased.json
  - results_distilbert-base-uncased.json


In [12]:
# Load all results
all_results = {}
for results_file in results_files:
    with open(results_file, "r") as f:
        results = json.load(f)
        model_name = results["model"]
        all_results[model_name] = results
        print(f"\nLoaded {model_name}")
        print(f"  Tasks: {results['tasks']}")
        print(f"  Number of seeds: {len(results['seeds'])}")

print(f"\nTotal models: {len(all_results)}")



Loaded bert-base-uncased
  Tasks: ['sst2', 'mrpc', 'qqp', 'stsb', 'cola', 'mnli', 'qnli', 'rte', 'wnli']
  Number of seeds: 5

Loaded distilbert-base-uncased
  Tasks: ['sst2', 'mrpc', 'qqp', 'stsb', 'cola', 'mnli', 'qnli', 'rte', 'wnli']
  Number of seeds: 5

Total models: 2


## Step 2: Create Summary Tables

Aggregate results by model and task.

In [5]:
# Create comprehensive results table
summary_rows = []

for model_name in sorted(all_results.keys()):
    model_results = all_results[model_name]
    print(f"\n{'='*80}")
    print(f"Model: {model_name}")
    print(f"{'='*80}")
    
    for task_name in sorted(model_results["results"].keys()):
        task_stats = model_results["results"][task_name]
        summary_rows.append({
            "Model": model_name,
            "Task": task_name.upper(),
            "Median": task_stats["median"],
            "Mean": task_stats["mean"],
            "Std": task_stats["std"],
            "Min": task_stats["min"],
            "Max": task_stats["max"],
            "All Scores": task_stats["scores"],
        })
        
        # Print task results
        print(f"  {task_name.upper():10} | "
              f"Median: {task_stats['median']:.4f} | "
              f"Mean: {task_stats['mean']:.4f} ± {task_stats['std']:.4f}")

# Create DataFrame
summary_df = pd.DataFrame(summary_rows)
print("\n\n" + "="*80)
print("SUMMARY TABLE - All Models and Tasks")
print("="*80)

# Format for display
display_df = summary_df[["Model", "Task", "Median", "Mean", "Std", "Min", "Max"]].copy()
for col in ["Median", "Mean", "Std", "Min", "Max"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")

print(display_df.to_string(index=False))



Model: bert-base-uncased
  COLA       | Median: 0.5756 | Mean: 0.5725 ± 0.0173
  MNLI       | Median: 0.8460 | Mean: 0.8460 ± 0.0011
  MRPC       | Median: 0.8652 | Mean: 0.8598 ± 0.0160
  QNLI       | Median: 0.9143 | Mean: 0.9141 ± 0.0016
  QQP        | Median: 0.9096 | Mean: 0.9097 ± 0.0019
  RTE        | Median: 0.6679 | Mean: 0.6679 ± 0.0245
  SST2       | Median: 0.9232 | Mean: 0.9243 ± 0.0019
  STSB       | Median: 0.8898 | Mean: 0.8880 ± 0.0065
  WNLI       | Median: 0.5493 | Mean: 0.5268 ± 0.0595

Model: distilbert-base-uncased
  COLA       | Median: 0.5109 | Mean: 0.5132 ± 0.0171
  MNLI       | Median: 0.8223 | Mean: 0.8218 ± 0.0022
  MRPC       | Median: 0.8505 | Mean: 0.8495 ± 0.0045
  QNLI       | Median: 0.8863 | Mean: 0.8865 ± 0.0038
  QQP        | Median: 0.9017 | Mean: 0.9018 ± 0.0005
  RTE        | Median: 0.5957 | Mean: 0.6014 ± 0.0347
  SST2       | Median: 0.9094 | Mean: 0.9080 ± 0.0032
  STSB       | Median: 0.8693 | Mean: 0.8691 ± 0.0012
  WNLI       | Median: 0

## Step 3: Pivot Tables for Model Comparison

Compare models side-by-side on each task.

In [6]:
if len(all_results) > 1:
    # Create pivot table: Tasks x Models (medians)
    pivot_data = []
    for _, row in summary_df.iterrows():
        pivot_data.append({
            "Task": row["Task"],
            f"{row['Model']} (Median)": f"{row['Median']:.4f}",
            f"{row['Model']} (Std)": f"{row['Std']:.4f}",
        })
    
    # Build comparison table manually
    tasks = summary_df["Task"].unique()
    models = summary_df["Model"].unique()
    
    comparison_data = []
    for task in sorted(tasks):
        row = {"Task": task}
        for model in sorted(models):
            task_model_data = summary_df[
                (summary_df["Task"] == task) & (summary_df["Model"] == model)
            ]
            if not task_model_data.empty:
                median = task_model_data.iloc[0]["Median"]
                std = task_model_data.iloc[0]["Std"]
                row[f"{model}"] = f"{median:.4f} ± {std:.4f}"
            else:
                row[f"{model}"] = "N/A"
        comparison_data.append(row)
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n\n" + "="*80)
    print("MODEL COMPARISON - Median ± Std")
    print("="*80)
    print(comparison_df.to_string(index=False))
    
    # Calculate performance gap (higher is better for most metrics)
    print("\n\n" + "="*80)
    print("PERFORMANCE GAP (Absolute Difference in Median Scores)")
    print("="*80)
    
    models_sorted = sorted(models)
    if len(models_sorted) >= 2:
        gap_rows = []
        for task in sorted(tasks):
            gap_row = {"Task": task}
            
            # Get scores for all models
            model_scores = {}
            for model in models_sorted:
                task_model_data = summary_df[
                    (summary_df["Task"] == task) & (summary_df["Model"] == model)
                ]
                if not task_model_data.empty:
                    model_scores[model] = task_model_data.iloc[0]["Median"]
            
            # Calculate differences
            if len(model_scores) >= 2:
                model_list = sorted(model_scores.keys())
                for i in range(len(model_list) - 1):
                    for j in range(i + 1, len(model_list)):
                        m1, m2 = model_list[i], model_list[j]
                        diff = abs(model_scores[m1] - model_scores[m2])
                        gap_row[f"|{m1} - {m2}|"] = f"{diff:.4f}"
            
            gap_rows.append(gap_row)
        
        gap_df = pd.DataFrame(gap_rows)
        print(gap_df.to_string(index=False))

else:
    print("\nOnly one model in results. Run finetuning for multiple models to see comparisons.")




MODEL COMPARISON - Median ± Std
Task bert-base-uncased distilbert-base-uncased
COLA   0.5756 ± 0.0173         0.5109 ± 0.0171
MNLI   0.8460 ± 0.0011         0.8223 ± 0.0022
MRPC   0.8652 ± 0.0160         0.8505 ± 0.0045
QNLI   0.9143 ± 0.0016         0.8863 ± 0.0038
 QQP   0.9096 ± 0.0019         0.9017 ± 0.0005
 RTE   0.6679 ± 0.0245         0.5957 ± 0.0347
SST2   0.9232 ± 0.0019         0.9094 ± 0.0032
STSB   0.8898 ± 0.0065         0.8693 ± 0.0012
WNLI   0.5493 ± 0.0595         0.5634 ± 0.0113


PERFORMANCE GAP (Absolute Difference in Median Scores)
Task |bert-base-uncased - distilbert-base-uncased|
COLA                                        0.0647
MNLI                                        0.0236
MRPC                                        0.0147
QNLI                                        0.0280
 QQP                                        0.0079
 RTE                                        0.0722
SST2                                        0.0138
STSB                           

## Step 4: Statistical Analysis

Compute overall statistics and compare with paper results.

In [7]:
print("="*80)
print("STATISTICAL SUMMARY BY MODEL")
print("="*80)

for model_name in sorted(all_results.keys()):
    model_results = all_results[model_name]
    all_medians = []
    all_stds = []
    
    for task_name, task_stats in model_results["results"].items():
        all_medians.append(task_stats["median"])
        all_stds.append(task_stats["std"])
    
    print(f"\n{model_name}")
    print(f"  Number of tasks: {len(all_medians)}")
    print(f"  Average median score: {np.mean(all_medians):.4f}")
    print(f"  Median of medians: {np.median(all_medians):.4f}")
    print(f"  Average std: {np.mean(all_stds):.4f}")
    print(f"  Min median: {np.min(all_medians):.4f}")
    print(f"  Max median: {np.max(all_medians):.4f}")


STATISTICAL SUMMARY BY MODEL

bert-base-uncased
  Number of tasks: 9
  Average median score: 0.7934
  Median of medians: 0.8652
  Average std: 0.0145
  Min median: 0.5493
  Max median: 0.9232

distilbert-base-uncased
  Number of tasks: 9
  Average median score: 0.7677
  Median of medians: 0.8505
  Average std: 0.0087
  Min median: 0.5109
  Max median: 0.9094


## Step 5: Detailed Per-Task Analysis

Show individual runs and variance per task.

In [8]:
# Show all runs for selected tasks
print("\n" + "="*80)
print("DETAILED RESULTS BY SEED")
print("="*80)

for model_name in sorted(all_results.keys()):
    model_results = all_results[model_name]
    print(f"\n{'='*80}")
    print(f"Model: {model_name}")
    print(f"Hyperparameters: {model_results['hyperparameters']}")
    print(f"{'='*80}")
    
    for task_name in sorted(model_results["results"].keys()):
        task_stats = model_results["results"][task_name]
        scores = task_stats["scores"]
        seeds = model_results["seeds"]
        
        print(f"\n  {task_name.upper()}")
        print(f"    Scores by seed: {[f'{s:.4f}' for s in scores]}")
        print(f"    Seeds:         {seeds}")
        print(f"    Median:        {task_stats['median']:.4f}")
        print(f"    Mean:          {task_stats['mean']:.4f}")
        print(f"    Std:           {task_stats['std']:.4f}")
        print(f"    Range:         [{task_stats['min']:.4f}, {task_stats['max']:.4f}]")



DETAILED RESULTS BY SEED

Model: bert-base-uncased
Hyperparameters: {'train_batch_size': 16, 'eval_batch_size': 32, 'num_epochs': 3, 'learning_rate': 2e-05, 'max_seq_length': 512}

  COLA
    Scores by seed: ['0.5502', '0.6007', '0.5591', '0.5767', '0.5756']
    Seeds:         [42, 19, 2005, 2026, 9999]
    Median:        0.5756
    Mean:          0.5725
    Std:           0.0173
    Range:         [0.5502, 0.6007]

  MNLI
    Scores by seed: ['0.8455', '0.8460', '0.8478', '0.8445', '0.8461']
    Seeds:         [42, 19, 2005, 2026, 9999]
    Median:        0.8460
    Mean:          0.8460
    Std:           0.0011
    Range:         [0.8445, 0.8478]

  MRPC
    Scores by seed: ['0.8750', '0.8725', '0.8652', '0.8309', '0.8554']
    Seeds:         [42, 19, 2005, 2026, 9999]
    Median:        0.8652
    Mean:          0.8598
    Std:           0.0160
    Range:         [0.8309, 0.8750]

  QNLI
    Scores by seed: ['0.9163', '0.9151', '0.9131', '0.9118', '0.9143']
    Seeds:         [42,

## Step 6: Comparison with Paper Results

The reproduction results are compared against these values from the paper.

### Results

Paper (Table 1):
| Model   | Score Avg. | CoLa | MNLI | MRPC | QNLI | QQP  | RTE  | SST-2 | STS-B | WNLI |
|---------|------|------|------|------|------|------|------|-------|-------|------|
|Bert-base (paper) | 79.5 | 56.3 | 86.7 | 88.6 | 91.8 | 89.6 | 69.3 | 92.7 | 89.0 | 53.5 | 
|DistilBERT (paper) | 77.0 | 51.3 | 82.2 | 87.5 | 89.2 | 88.5 | 59.9 | 91.3 | 86.9 | 56.3 |

Our reproduced results:

| Model   | Score Avg. | CoLa | MNLI | MRPC | QNLI | QQP  | RTE  | SST-2 | STS-B | WNLI |
|---------|------|------|------|------|------|------|------|-------|-------|------|
|Bert-base (ours)| 79.4 | 57.6 | 84.6 | 86.5 | 91.4 | 91.0 | 66.8 | 92.3 |  89.0 | 55.0 | 
|DistilBERT (ours)| 76.8 | 51.1 | 82.2 | 85.1 | 88.6 | 90.2 | 59.6 | 91.0  | 87.0 | 56.3 |  



In [10]:
# Paper results for reference
paper_results = {
    "bert-base-uncased": {
        "sst2": 92.7,
        "mrpc": 88.6,
        "qqp": 89.6,
        "stsb": 89.0,
        "cola": 56.3,
        "mnli": 86.7,
        "qnli": 91.8,
        "rte": 69.3,
    },
    "distilbert-base-uncased": {
        "sst2": 91.3,
        "mrpc": 87.5,
        "qqp": 88.5,
        "stsb": 86.9,
        "cola": 51.3,
        "mnli": 82.2,
        "qnli": 89.2,
        "rte": 59.9,
    },
}

# Compare our results with paper
print("\n" + "="*80)
print("COMPARISON WITH PAPER RESULTS (Sanh et al., 2019)")
print("="*80)

for model_name in sorted(all_results.keys()):
    if model_name in paper_results:
        print(f"\n{'='*80}")
        print(f"Model: {model_name}")
        print(f"{'='*80}")
        print(f"{'Task':<12} {'Our Median':<12} {'Paper':<12} {'Diff':<12}")
        print(f"{'-'*48}")
        
        our_results = all_results[model_name]["results"]
        paper_model = paper_results[model_name]
        
        diffs = []
        for task_name in sorted(our_results.keys()):
            our_median = our_results[task_name]["median"] * 100  # Convert to percentage
            paper_value = paper_model.get(task_name, None)
            
            if paper_value is not None:
                diff = our_median - paper_value
                diffs.append(diff)
                diff_str = f"{diff:+.2f}%" if abs(diff) < 100 else "N/A"
                print(f"{task_name:<12} {our_median:>10.2f}% {paper_value:>10.1f}% {diff_str:>10}")
        
        if diffs:
            print(f"{'-'*48}")
            print(f"{'Average diff':<12} {f'{np.mean(diffs):+.2f}%':>34}")



COMPARISON WITH PAPER RESULTS (Sanh et al., 2019)

Model: bert-base-uncased
Task         Our Median   Paper        Diff        
------------------------------------------------
cola              57.56%       56.3%     +1.26%
mnli              84.60%       86.7%     -2.10%
mrpc              86.52%       88.6%     -2.08%
qnli              91.43%       91.8%     -0.37%
qqp               90.96%       89.6%     +1.36%
rte               66.79%       69.3%     -2.51%
sst2              92.32%       92.7%     -0.38%
stsb              88.98%       89.0%     -0.02%
------------------------------------------------
Average diff                             -0.61%

Model: distilbert-base-uncased
Task         Our Median   Paper        Diff        
------------------------------------------------
cola              51.09%       51.3%     -0.21%
mnli              82.23%       82.2%     +0.03%
mrpc              85.05%       87.5%     -2.45%
qnli              88.63%       89.2%     -0.57%
qqp             

## Step 7: Save Consolidated Results

Export all results to a single JSON file for archiving.

In [ ]:
# Save consolidated results
consolidated_path = Path("./outputs/glue_benchmark/consolidated_results.json")
consolidated_path.parent.mkdir(parents=True, exist_ok=True)

consolidated = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "models_results": all_results,
    "paper_reference": paper_results,
    "summary_statistics": {},
}

# Add summary statistics
for model_name in sorted(all_results.keys()):
    model_results = all_results[model_name]
    all_medians = []
    
    for task_name, task_stats in model_results["results"].items():
        all_medians.append(task_stats["median"])
    
    consolidated["summary_statistics"][model_name] = {
        "average_median_score": float(np.mean(all_medians)),
        "median_of_medians": float(np.median(all_medians)),
        "min_median": float(np.min(all_medians)),
        "max_median": float(np.max(all_medians)),
    }

with open(consolidated_path, "w") as f:
    json.dump(consolidated, f, indent=2, default=str)

print(f"\nConsolidated results saved to: {consolidated_path}")



✓ Consolidated results saved to: outputs\glue_benchmark\consolidated_results.json


## Summary

This evaluation notebook provides:

1. **Results Aggregation**: Loads all fine-tuning results from multiple seeds
2. **Summary Statistics**: Median, mean, std, min, max for each task
3. **Model Comparison**: Side-by-side comparison of different models
4. **Paper Comparison**: Validates results against published DistilBERT paper results
5. **Detailed Analysis**: Per-seed breakdown and variance analysis
6. **Result Export**: Consolidated JSON for archiving and sharing

